# Member 1 — Baseline and Logistic Regression

Owner: IT24101668

Purpose: Establish baseline performance and train Logistic Regression model.


## Cell 1 — Load the validated dataset

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load cleaned dataset and saved split
data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

# Exclude target and unavailable or deferred features
excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X = df.drop(columns=excluded)
y = df["is_canceled"]

# Reuse the approved train/test split
X_train = X.iloc[splits["train_idx"]].copy()
X_test = X.iloc[splits["test_idx"]].copy()

y_train = y.iloc[splits["train_idx"]].copy()
y_test = y.iloc[splits["test_idx"]].copy()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Selected features:", X_train.shape[1])
print("Training cancellation rate:", f"{y_train.mean():.2%}")


Training shape: (95334, 21)
Testing shape: (23876, 21)
Selected features: 21
Training cancellation rate: 37.01%


## Cell 2 — DummyClassifier Baseline

In [2]:

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score

# Keep identical feature combinations in the same fold
groups_train = pd.util.hash_pandas_object(X_train, index=False)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create baseline model
baseline = DummyClassifier(strategy="prior")

# Evaluate using training-only cross-validation
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring
)

print("DummyClassifier - 5-Fold Cross-Validation")

for metric in scoring:
    score = baseline_scores[f"test_{metric}"].mean()
    print(f"{metric}: {score:.4f}")


DummyClassifier - 5-Fold Cross-Validation
accuracy: 0.6302
precision: 0.0000
recall: 0.0000
f1: 0.0000
roc_auc: 0.5000
pr_auc: 0.3698


## Cell 3 — Logistic Regression Model

In [3]:

import sys
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

# Import the shared preprocessing pipeline
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

# Create Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessing", build_preprocessor(X_train)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    ))
])

# Evaluate using the same 5-fold cross-validation
logistic_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

print("Logistic Regression - 5-Fold Cross-Validation")

for metric in scoring:
    score = logistic_scores[f"test_{metric}"].mean()
    print(f"{metric}: {score:.4f}")


Logistic Regression - 5-Fold Cross-Validation
accuracy: 0.7774
precision: 0.7541
recall: 0.5909
f1: 0.6622
roc_auc: 0.8421
pr_auc: 0.7906


## Cell 4 — Compare both models

In [4]:

# Compare baseline and Logistic Regression
comparison = pd.DataFrame({
    "DummyClassifier": {
        metric: baseline_scores[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Logistic Regression": {
        metric: logistic_scores[f"test_{metric}"].mean()
        for metric in scoring
    }
}).T

display(comparison.round(4))


,accuracy,precision,recall,f1,roc_auc,pr_auc
DummyClassifier,0.6302,0.0000,0.0000,0.0000,0.5000,0.3698
Logistic Regression,0.7774,0.7541,0.5909,0.6622,0.8421,0.7906


## Cell 5 — Save Baseline and Logistic Regression Results

In [5]:

# Save cross-validation results for both models
output_dir = Path("../../results/metrics")
output_dir.mkdir(parents=True, exist_ok=True)

models = {
    "dummy_classifier": baseline_scores,
    "logistic_regression": logistic_scores
}

for name, scores in models.items():
    results = pd.DataFrame({
        metric: scores[f"test_{metric}"]
        for metric in scoring
    })

    results.index = [f"Fold {i}" for i in range(1, 6)]
    results.loc["Mean"] = results.mean()

    results.to_csv(output_dir / f"{name}_cv.csv")

    print(f"Saved: {name}_cv.csv")

print("\nBoth model results saved successfully.")


Saved: dummy_classifier_cv.csv
Saved: logistic_regression_cv.csv

Both model results saved successfully.


## Baseline and Logistic Regression Results

The DummyClassifier achieved 63.02% accuracy but failed to identify cancellations, resulting in zero precision, recall and F1-score.

Logistic Regression significantly improved prediction performance, achieving 77.74% accuracy, 75.41% precision, 59.09% recall, and an F1-score of 0.6622. Its ROC-AUC of 0.8421 and PR-AUC of 0.7906 indicate useful predictive discrimination.

However, approximately 41% of actual cancellations were not identified at the default classification threshold. Improving cancellation recall will therefore be an important consideration when comparing alternative models.

All results were obtained using five-fold group-aware cross-validation on the training dataset. The held-out test dataset remains unused.

## Cell 6 — Configure Hyperparameter Search

In [6]:

from sklearn.model_selection import GridSearchCV

# Small parameter grid
param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"]
}

# Use 3 folds for faster tuning
tuning_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

logistic_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=param_grid,
    scoring="average_precision",
    cv=tuning_cv,
    n_jobs=1
)

print("Hyperparameter search configured.")
print("Parameter combinations:", 6)


Hyperparameter search configured.
Parameter combinations: 6


## Cell 7 — Run Hyperparameter Tuning

In [7]:

# Run hyperparameter tuning on training data only
logistic_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Logistic Regression tuning completed successfully.")


Logistic Regression tuning completed successfully.


## Cell 8 — Display Best Hyperparameters and Tuning Score

In [8]:

# Display the best tuning results
print("Logistic Regression - Hyperparameter Tuning Results\n")

print("Best hyperparameters:")
for param, value in logistic_search.best_params_.items():
    print(f"{param}: {value}")

print("\nBest cross-validation PR-AUC:",
      f"{logistic_search.best_score_:.4f}")


Logistic Regression - Hyperparameter Tuning Results

Best hyperparameters:
classifier__C: 1
classifier__class_weight: None

Best cross-validation PR-AUC: 0.7907


## Cell 9 — Review All Hyperparameter Combinations

In [9]:

# Compare all tested parameter combinations
tuning_results = pd.DataFrame(logistic_search.cv_results_)

columns = [
    "param_classifier__C",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score"
]

display(
    tuning_results[columns]
    .sort_values("mean_test_score", ascending=False)
    .round(4)
)


,param_classifier__C,param_classifier__class_weight,mean_test_score,std_test_score
2,1.0,None,0.7907,0.0103
4,10.0,None,0.7905,0.0102
0,0.1,None,0.7898,0.0105
3,1.0,balanced,0.7878,0.0100
5,10.0,balanced,0.7878,0.0098
1,0.1,balanced,0.7872,0.0100


## Cell 10 — Save Hyperparameter Tuning Results

In [10]:

# Save Logistic Regression tuning results
output = Path("../../results/metrics/logistic_regression_tuning.csv")

output.parent.mkdir(parents=True, exist_ok=True)

tuning_results[[
    "param_classifier__C",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values(
    "rank_test_score"
).to_csv(output, index=False)

print("Tuning results saved:", output)
print("Best parameters:", logistic_search.best_params_)


Tuning results saved: ..\..\results\metrics\logistic_regression_tuning.csv
Best parameters: {'classifier__C': 1, 'classifier__class_weight': None}


## Hyperparameter Tuning Conclusion

Six Logistic Regression configurations were compared using three-fold group-aware cross-validation, with PR-AUC (average precision) as the selection metric. The best parameters were `C=1` and `class_weight=None`, with a tuning PR-AUC of **0.7907**.

**Final decision:** Retain the original Logistic Regression configuration because tuning selected the same settings. The original five-fold cross-validation results remain the comparison reference (F1-score **0.6622**, PR-AUC **0.7906**). The three-fold tuning score and five-fold evaluation score should not be treated as directly comparable improvements. The held-out test set has not been used.